## Initializing polymer systems using a DPD potential
This notebook walks through the PhantomWalk functions for packing linear polymers in a box. The polymers are first placed in a cubic box using a random walk. Then a short HOOMD simulation is run with the soft force potential of Dissipative Particle Dynamics. The simulation ends when the pair energy from the DPD potential reaches a stable state, as checked with an autocorrelation function.

In [1]:
import matplotlib
import numpy as np  
import gsd, gsd.hoomd 
import hoomd 
from cmeutils.sampling import is_equilibrated
import time
import freud
import matplotlib_inline
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.style.use("ggplot")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [2]:
def initialize_snapshot_rand_walk(num_pol, num_mon, density=0.85, bond_length=1.0, buffer=0.1):
    """Create a HOOMD snapshot containing random-walk polymer chains.

    Polymer chains are initialized as random walks in a
    cubic simulation box with periodic boundary conditions.
    Bond, angle, and dihedral connectivity are generated
    for each chain.

    Parameters
    ----------
    num_pol : int
        Number of polymer chains in the simulation.

    num_mon : int
        Number of monomers per polymer chain.

    density : float, optional, default 0.85
        Target particle number density used to determine
        the cubic simulation box size.

    bond_length : float, optional, default 1.0
        Bond length between connected monomers.

    buffer : float, optional, default 0.1
        This buffer is reserved to reduce the size of the box length for the random walk.

    Returns
    -------
    gsd.hoomd.Frame
        HOOMD snapshot containing:
        - particle positions
        - bond connectivity
        - angle connectivity
        - dihedral connectivity
        - particle and topology types
        - simulation box dimensions

    """   
    N = num_pol * num_mon
    L = np.cbrt(N / density)  # Calculate box size based on density
    positions = np.zeros((N, 3))
    for i in range(num_pol):
        start = i * num_mon
        positions[start] = np.random.uniform(low=(-L/2),high=(L/2),size=3)
        for j in range(num_mon - 1):
            delta = np.random.uniform(low=(-bond_length/2),high=(bond_length/2),size=3)
            delta /= np.linalg.norm(delta)*bond_length
            positions[start+j+1] = positions[start+j] + delta
    positions = pbc(positions,[L,L,L])
    bonds = []
    for i in range(num_pol):
        start = i * num_mon
        for j in range(num_mon - 1):
            bonds.append([start + j, start + j + 1])
    angles = []
    for i in range(num_pol):
        start = i * num_mon
        for j in range(num_mon - 2):
            angles.append([start + j, start + j + 1,start + j + 2])
    dihedrals = []
    for i in range(num_pol):
        start = i * num_mon
        for j in range(num_mon - 3):
            dihedrals.append([start + j, start + j + 1,start + j + 2,start + j + 3])
    bonds = np.array(bonds)
    frame = gsd.hoomd.Frame()
    frame.particles.types = ['A']
    frame.particles.N = N
    frame.particles.position = positions
    frame.bonds.N = len(bonds)
    frame.bonds.group = bonds
    frame.bonds.types = ['b']
    frame.angles.N = (num_mon - 2)*num_pol
    frame.angles.types = ["A-A-A"]
    frame.angles.group = angles
    frame.dihedrals.N = (num_mon - 3)*num_pol
    frame.dihedrals.types = ["A-A-A-A"]
    frame.dihedrals.group = dihedrals
    frame.configuration.box = [L, L, L, 0, 0, 0]
    return frame
    
def pbc(d,box):
    """Apply periodic boundary conditions to particle positions.

    Parameters
    ----------
    d : np.ndarray
        Particle position array with shape (N, 3).

    box : list[float]
        Simulation box lengths in the x, y, and z
        directions.

    Returns
    -------
    np.ndarray
        Particle positions wrapped into the simulation box.

    """
    for i in range(3):
        a = d[:,i]
        pos_max = np.max(a)
        pos_min = np.min(a)
        while pos_max > box[i]/2 or pos_min < -box[i]/2:
            a[a < -box[i]/2] += box[i]
            a[a >  box[i]/2] -= box[i]
            pos_max = np.max(a)
            pos_min = np.min(a)
    return d

def check_pair_energy(step_cut):
    """Check whether the pair interaction energy has equilibrated.

    Pair energies are read from the HOOMD log file and analyzed
    using pymbar timeseries equilibration detection.

    Parameters
    ----------
    step_cut : int
        Number of initial simulation steps to discard before
        performing equilibration analysis.

    Returns
    -------
    bool
        True if the pair energy timeseries is determined
        to be equilibrated, otherwise False.

    """
    log = np.genfromtxt("log.txt", names=True)
    pairs = log["mdpairDPDenergy"]
    shrink_cut = step_cut
    equil, t0, g, neff = is_equilibrated(data=pairs[shrink_cut:], threshold_neff=50) 
    if equil:
        return True
    else:
        return False
    
def add_hoomd_writers(sim):
    """Add GSD trajectory and log writers to a HOOMD simulation.

    This function creates:
    - a GSD trajectory writer for particle configurations
    - a table logger for thermodynamic and force quantities
    - thermodynamic compute operations for system properties

    Parameters
    ----------
    sim : hoomd.Simulation
        HOOMD simulation object to which writers and
        computes will be attached.

    Returns
    -------
    None
        This function modifies the simulation object in place
        and does not return a value.

    """
    gsd_logger = hoomd.logging.Logger(
        categories=["scalar", "string", "sequence"]
    )
    logger = hoomd.logging.Logger(categories=["scalar", "string"])
    gsd_logger.add(sim, quantities=["timestep", "tps"])
    logger.add(sim, quantities=["timestep", "tps"])
    thermo_props = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
    sim.operations.computes.append(thermo_props)
    log_quantities = [
            "kinetic_temperature",
            "potential_energy",
            "kinetic_energy",
            "volume",
            "pressure",
            "pressure_tensor",
        ]
    gsd_logger.add(thermo_props, quantities=log_quantities)
    logger.add(thermo_props, quantities=log_quantities)

    for f in sim.operations.integrator.forces:
        logger.add(f, quantities=["energy"])
        gsd_logger.add(f, quantities=["energy"])

    gsd_writer = hoomd.write.GSD(
        filename='trajectory.gsd',
        trigger=hoomd.trigger.Periodic(int(10)),
        mode="wb",
        dynamic=["momentum", "property"],
        filter=hoomd.filter.All(),
        logger=gsd_logger,
    )
    gsd_writer.maximum_write_buffer_size = 64 * 1024 * 1024

    table_file = hoomd.write.Table(
        output=open('log.txt', mode="w", newline="\n"),
        trigger=hoomd.trigger.Periodic(period=int(10)),
        logger=logger,
        max_header_len=None,
    )
    sim.operations.writers.append(gsd_writer)
    sim.operations.writers.append(table_file)

def run_nvt_dpd_simulation(
    A=1000,
    gamma=1000,
    k=1000,
    angle_k=3.0,
    angle_t0=1.0,
    dihedral_k=3.0,
    dihedral_d=-1,
    dihedral_n=3,
    dihedral_phi0=0,
    num_pol=100,
    num_mon=10,
    kT=1.0,
    r_cut=1.15,
    bond_l=1.0,
    dt=0.001,
    density=0.8,
    particle_spacing=1.1
):
    """Run a single polymer equilibration simulation in HOOMD-blue.

    Parameters
    ----------
    A : float, optional, default 1000
        Conservative force parameter for the DPD interaction.

    gamma : float, optional, default 1000
        Dissipative force coefficient for the DPD interaction.

    k : float, optional, default 1000
        Harmonic bond spring constant.

    angle_k : float, optional, default 3.0
        Harmonic angle force constant.

    angle_t0 : float, optional, default 1.0
        Equilibrium bond angle in radians for the harmonic
        angle potential.

    dihedral_k : float, optional, default 3.0
        Force constant for the periodic dihedral potential.

    dihedral_d : int, optional, default -1
        Sign factor for the periodic dihedral potential.

    dihedral_n : int, optional, default 3
        Periodicity of the dihedral potential.

    dihedral_phi0 : float, optional, default 0
        Phase shift angle for the periodic dihedral
        potential in radians.

    num_pol : int, optional, default 100
        Number of polymer chains in the simulation.

    num_mon : int, optional, default 10
        Number of monomers per polymer chain.

    kT : float, optional, default 1.0
        Thermal energy used by the DPD thermostat.

    r_cut : float, optional, default 1.15
        Cutoff radius for DPD interactions.

    bond_l : float, optional, default 1.0
        Equilibrium bond length for harmonic bonds.

    dt : float, optional, default 0.001
        Simulation integration timestep.

    density : float, optional, default 0.8
        Target particle number density.

    particle_spacing : float, optional, default 1.1
        Maximum acceptable bond length during equilibration.

    Returns
    -------
    tuple
        Tuple containing:
        - final simulation snapshot
        - elapsed runtime in seconds

        If equilibration fails due to timeout,
        runtime is returned as 0.

    """
    print(num_pol * num_mon)

    print(
        f"\nRunning with A={A}, gamma={gamma}, k={k}, "
        f"num_pol={num_pol}, num_mon={num_mon}"
    )

    start_time = time.perf_counter()

    frame = initialize_snapshot_rand_walk(
        num_pol,
        num_mon,
        density=density
    )

    build_stop = time.perf_counter()

    print("Total build time: ", build_stop - start_time)

    integrator = hoomd.md.Integrator(dt=dt)

    harmonic = hoomd.md.bond.Harmonic()
    harmonic.params["b"] = dict(r0=bond_l, k=k)
    integrator.forces.append(harmonic)

    harmonic_angle = hoomd.md.angle.Harmonic()
    harmonic_angle.params["A-A-A"] = dict(
        k=angle_k,
        t0=angle_t0
    )
    integrator.forces.append(harmonic_angle)

    dihedral = hoomd.md.dihedral.Periodic()
    dihedral.params["A-A-A-A"] = dict(
        k=dihedral_k,
        d=dihedral_d,
        n=dihedral_n,
        phi0=dihedral_phi0
    )
    integrator.forces.append(dihedral)

    simulation = hoomd.Simulation(
        device=hoomd.device.auto_select(),
        seed=np.random.randint(65535)
    )

    simulation.operations.integrator = integrator
    simulation.create_state_from_snapshot(frame)

    const_vol = hoomd.md.methods.ConstantVolume(
        filter=hoomd.filter.All()
    )

    integrator.methods.append(const_vol)

    nlist = hoomd.md.nlist.Cell(buffer=0.4)
    simulation.operations.nlist = nlist

    DPD = hoomd.md.pair.DPD(
        nlist,
        default_r_cut=r_cut,
        kT=kT
    )

    DPD.params[('A', 'A')] = dict(
        A=A,
        gamma=gamma
    )

    integrator.forces.append(DPD)

    add_hoomd_writers(sim=simulation)

    simulation.run(0)
    simulation.run(1000)

    for writer in simulation.operations.writers:
        if hasattr(writer, "flush"):
            writer.flush()

    snap = simulation.state.get_snapshot()

    shrink_cut = 5

    while not check_pair_energy(shrink_cut):
        check_time = time.perf_counter()

        if (check_time - start_time) > 60:
            return num_pol * num_mon, 0

        simulation.run(1000)

        for writer in simulation.operations.writers:
            if hasattr(writer, "flush"):
                writer.flush()

        snap = simulation.state.get_snapshot()

        shrink_cut += 50

    end_time = time.perf_counter()

    return snap, end_time - start_time

In [3]:
dpd_final_frame,s = run_nvt_dpd_simulation(A=1000,gamma=800,k=20000,num_pol=1,num_mon=5000)
print(f"Finished in time = {s:.2f}s")

5000

Running with A=1000, gamma=800, k=20000, num_pol=1, num_mon=5000
Total build time:  0.03012791700894013
Finished in time = 15.40s


## Run a Lennard Jones Simulation from the Final DPD Frame

In [4]:
def run_lj_simulation(
    dpd_final_frame,
    random_seed=24,
    dt=0.001,
    lj_epsilon=1.0,
    lj_sigma=1.0,
    lj_r_cut=1.2,
    fene_k=30,
    fene_r0=1.05,
    fene_epsilon=1.0,
    fene_sigma=1.0,
    fene_delta=0,
    angle_k=3.0,
    angle_t0=1.0,
    dihedral_k=3.0,
    dihedral_d=-1,
    dihedral_n=3,
    dihedral_phi0=0
):
    """Run an LJ + FENE + angle + dihedral equilibration simulation in HOOMD-blue.

    Parameters
    ----------
    dpd_final_frame : gsd.hoomd.Frame
        Initial configuration used to start the LJ simulation.

    random_seed : int, optional, default 24
        Random seed for reproducibility.

    dt : float, optional, default 0.001
        Integration timestep.

    lj_epsilon : float, optional, default 1.0
        Lennard-Jones interaction strength.

    lj_sigma : float, optional, default 1.0
        Lennard-Jones particle size parameter.

    lj_r_cut : float, optional, default 1.2
        Lennard-Jones cutoff radius.

    fene_k : float, optional, default 30
        FENE bond spring constant.

    fene_r0 : float, optional, default 1.05
        FENE maximum bond extension parameter.

    fene_epsilon : float, optional, default 1.0
        FENE-WCA epsilon parameter.

    fene_sigma : float, optional, default 1.0
        FENE-WCA sigma parameter.

    fene_delta : float, optional, default 0
        FENE potential shift parameter.

    angle_k : float, optional, default 3.0
        Harmonic angle force constant.

    angle_t0 : float, optional, default 1.0
        Equilibrium bond angle (radians).

    dihedral_k : float, optional, default 3.0
        Dihedral force constant.

    dihedral_d : int, optional, default -1
        Dihedral sign parameter.

    dihedral_n : int, optional, default 3
        Dihedral periodicity.

    dihedral_phi0 : float, optional, default 0
        Dihedral phase offset (radians).

    Returns
    -------
    hoomd.Simulation
        HOOMD simulation object after short equilibration run.
    """

    forces = []

    # Pair force (LJ)
    nlist = hoomd.md.nlist.Cell(buffer=0.40, exclusions=["bond"])
    lj = hoomd.md.pair.LJ(nlist=nlist)
    lj.params[('A', 'A')] = dict(epsilon=lj_epsilon, sigma=lj_sigma)
    lj.r_cut[('A', 'A')] = lj_r_cut
    forces.append(lj)

    # FENE bonds
    fene_bond = hoomd.md.bond.FENEWCA()
    fene_bond.params['b'] = dict(
        k=fene_k,
        r0=fene_r0,
        epsilon=fene_epsilon,
        sigma=fene_sigma,
        delta=fene_delta,
    )
    forces.append(fene_bond)

    # Angle potential
    harmonic_angle = hoomd.md.angle.Harmonic()
    harmonic_angle.params["A-A-A"] = dict(k=angle_k, t0=angle_t0)
    forces.append(harmonic_angle)

    # Dihedral potential
    dihedral = hoomd.md.dihedral.Periodic()
    dihedral.params["A-A-A-A"] = dict(
        k=dihedral_k,
        d=dihedral_d,
        n=dihedral_n,
        phi0=dihedral_phi0
    )
    forces.append(dihedral)

    # Integrator
    integrator_lj = hoomd.md.Integrator(dt=dt)
    integrator_lj.forces = forces

    integrator_lj.methods.append(
        hoomd.md.methods.ConstantVolume(filter=hoomd.filter.All())
    )

    # Simulation
    LJ_sim = hoomd.Simulation(
        device=hoomd.device.auto_select(),
        seed=random_seed
    )

    LJ_sim.create_state_from_snapshot(snapshot=dpd_final_frame)
    LJ_sim.operations.integrator = integrator_lj

    # Use your shared writer setup
    add_hoomd_writers(sim=LJ_sim)

    # Run short equilibration
    LJ_sim.run(0)
    LJ_sim.run(100)

    # Flush outputs
    for writer in LJ_sim.operations.writers:
        if hasattr(writer, "flush"):
            writer.flush()

    print("LJ simulation finished.")

    return LJ_sim

In [5]:
run_lj_simulation(dpd_final_frame=dpd_final_frame)

LJ simulation finished.
